# Policy ranking under different patience hazard shapes

This notebook explores how the performance of different service policies changes depending on the **shape of the patience hazard rate**.

The goal is to illustrate a key idea behind TIQ-style policies:

Different hazard structures favor different queue disciplines.

## System setup

We simulate a single-class overloaded system:

$$
N = 10, \quad \lambda = 20, \quad \mu = 1
$$

Since

$$
\lambda > N\mu
$$

the system is overloaded and abandonment stabilizes the queue.

Arrival and service processes are exponential, while patience distributions are varied to produce different hazard shapes.

## Patience distributions

We test three patience distributions representing different hazard behaviors:

Decreasing hazard rate (DHR)

$$
\text{Weibull}(k = 0.7)
$$

Constant hazard rate

$$
\text{Exponential}(\theta = 0.5)
$$

Increasing hazard rate (IHR)

$$
\text{Weibull}(k = 1.5)
$$

These three cases illustrate how abandonment risk changes with waiting time.

## Policies compared

We compare three service disciplines:

FCFS  
First-come first-served.

LCFS  
Last-come first-served.

TIQ  
A threshold index queue with thresholds

$$
w_\ell = 0.2, \quad w_h = 1.5.
$$

These thresholds are not optimized.  
They simply illustrate how TIQ behaves under different hazard structures.

## Expectation

Theoretical work on abandonment queues suggests that hazard shape affects which policy performs best.

Intuitively:

- If the hazard is decreasing, older customers are less likely to abandon, so policies that serve newer arrivals may perform well.
- If the hazard is increasing, customers become more likely to abandon as they wait, so prioritizing long waits becomes beneficial.
- The exponential case sits between these extremes.

## Interpreting the output

The table reports the following quantities:

- $\lambda_{\text{hat}}$: Estimated arrival rate.

- served_rate: Rate of service completions.

- abandon_rate: Rate of abandonment.

- $L$: Average number of customers in the system.

- $W$: Average time in system for served customers.

By comparing these metrics across patience distributions and policies, we can see how hazard shape changes policy performance.

In [ ]:
import pandas as pd

from gi_gi_n_gi_multiclass.core.engine import run_sim
from gi_gi_n_gi_multiclass.core.types import SimConfig
from gi_gi_n_gi_multiclass.dists.common import Exponential, Weibull
from gi_gi_n_gi_multiclass.policies.fcfs import FCFS
from gi_gi_n_gi_multiclass.policies.lcfs import LCFS
from gi_gi_n_gi_multiclass.policies.tiq import TIQ
from gi_gi_n_gi_multiclass.metrics.basic import summary_table


In [1]:

N = 10
lambda_rate = 20.0
mu_rate = 1.0
warmup_time = 500.0
run_time = 2000.0
seed = 123

interarrival = Exponential(rate=lambda_rate)
service = Exponential(rate=mu_rate)

patience_cases = {
    "DHR_Weibull_k0.7": Weibull(shape=0.7, scale=2.0),
    "Exp_rate0.5": Exponential(rate=0.5),
    "IHR_Weibull_k1.5": Weibull(shape=1.5, scale=2.0),
}

policies = {
    "FCFS": FCFS(),
    "LCFS": LCFS(),
    "TIQ_0.2_1.5": TIQ(w_l=0.2, w_h=1.5),
}

rows = []
for pat_name, patience in patience_cases.items():
    for pol_name, policy in policies.items():
        cfg = SimConfig(
            n_servers=N,
            interarrival=interarrival,
            service=service,
            patience=patience,
            policy=policy,
            seed=seed,
            warmup_time=warmup_time,
            run_time=run_time,
            collect_event_log=False,
        )
        res = run_sim(cfg)
        tbl = summary_table(res)
        tbl["patience_case"] = pat_name
        tbl["policy_name"] = pol_name
        rows.append(tbl)

out = pd.concat(rows, ignore_index=True)

cols = [
    "patience_case",
    "policy_name",
    "lambda_hat",
    "served_rate",
    "abandon_rate",
    "L_timeavg",
    "W_mean",
]
out[cols].sort_values(["patience_case", "policy_name"])

,patience_case,policy_name,lambda_hat,served_rate,abandon_rate,L_timeavg,W_mean
0,DHR_Weibull_k0.7,FCFS,20.0135,10.0095,10.0010,26.309276,2.167692
1,DHR_Weibull_k0.7,LCFS,20.0135,9.9970,10.0095,33.261004,1.130919
2,DHR_Weibull_k0.7,TIQ_0.2_1.5,20.0135,10.0395,9.9700,27.122726,2.146394
3,Exp_rate0.5,FCFS,20.0135,10.0160,9.9915,29.870155,2.328707
4,Exp_rate0.5,LCFS,20.0135,10.0570,9.9500,29.976947,1.105338
5,Exp_rate0.5,TIQ_0.2_1.5,20.0135,9.9930,10.0145,30.109732,2.265048
6,IHR_Weibull_k1.5,FCFS,20.0135,10.0170,9.9885,33.926286,2.489388
7,IHR_Weibull_k1.5,LCFS,20.0135,10.0190,9.9850,28.663770,1.101985
8,IHR_Weibull_k1.5,TIQ_0.2_1.5,20.0135,10.0185,9.9885,33.163050,2.346452


## Takeaway

This experiment shows that the effectiveness of a scheduling policy depends on the patience hazard structure.

This observation motivates the use of hazard-aware policies such as TIQ and its multiclass extensions.

Later notebooks will explore how these ideas extend to heterogeneous systems and to the mTIQ policy derived in Bassamboo et al.